# 01 — EDA: conhecendo a base global de saúde

Objetivo: documentar o que a ABT (`gold/abt_country_year.parquet`) contém, o que é
confiável, o que falta e como o missing se distribui. Painel país × ano, 1990–2023,
World Bank (primária) + WHO GHO (secundária) e o proxy de cobertura UHC.

In [1]:
import sys; sys.path.insert(0, ".")
import pandas as pd
import plotly.express as px
from src.analysis import eda

pd.set_option("display.max_columns", 40)
abt = eda.carregar_abt()
print(abt.shape)
abt.head(3)

(7378, 38)


,country_code,country_name,region,income_level,year,beds_per_1000,child_mortality,doctors_per_1000,dpt_imm_pct,fertility,gdp_per_capita,health_exp_gdp,health_exp_per_capita,latitude,lending_type,life_expectancy,longitude,maternal_mortality,measles_imm_pct,ncd_mortality_30_70,nurses_per_1000,out_of_pocket,population,public_health_exp_gdp,road_traffic_death_rate,sanitation_basic,sanitation_safely,tb_incidence,uhc_sci,urban_pct,water_basic,water_safely,uhc_index,treated,treat_year,post,always_treated,never_treated
0,ABW,Aruba,Latin America & Caribbean,High income,1990,NaN,NaN,NaN,NaN,2.345,12187.536361,NaN,NaN,12.5167,Not classified,72.546,-70.0167,NaN,NaN,NaN,NaN,NaN,62753.0,NaN,NaN,NaN,NaN,NaN,NaN,65.432816,NaN,NaN,NaN,None,<NA>,None,<NA>,None
1,ABW,Aruba,Latin America & Caribbean,High income,1991,NaN,NaN,NaN,NaN,2.362,13233.990517,NaN,NaN,12.5167,Not classified,72.592,-70.0167,NaN,NaN,NaN,NaN,NaN,65896.0,NaN,NaN,NaN,NaN,NaN,NaN,65.400922,NaN,NaN,NaN,None,<NA>,None,<NA>,None
2,ABW,Aruba,Latin America & Caribbean,High income,1992,NaN,NaN,NaN,NaN,2.353,13892.605143,NaN,NaN,12.5167,Not classified,72.717,-70.0167,NaN,NaN,NaN,NaN,NaN,69005.0,NaN,NaN,NaN,NaN,NaN,NaN,65.390589,NaN,NaN,NaN,None,<NA>,None,<NA>,None


## 1. Cobertura e granularidade

In [2]:
resumo = eda.resumo_cobertura(abt)
print(f"linhas={resumo['rows']}  paises={resumo['countries']}  "
      f"anos={resumo['year_min']}-{resumo['year_max']}  regioes={resumo['n_regions']}")

cov = pd.DataFrame(resumo["coverage"]).T.sort_values("missing_rate")
cov

linhas=7378  paises=217  anos=1990-2023  regioes=7


,n,missing_rate
life_expectancy,7378.0,0.0000
population,7378.0,0.0000
fertility,7378.0,0.0000
urban_pct,7378.0,0.0000
gdp_per_capita,7015.0,0.0492
child_mortality,6664.0,0.0968
dpt_imm_pct,6432.0,0.1282
measles_imm_pct,6424.0,0.1293
uhc_index,5564.0,0.2459
water_basic,5007.0,0.3214


### Leitura

- O painel é **balanceado no papel** (217 países × 34 anos), mas os indicadores têm
  cobertura bem diferente: alvos populacionais (LE, mortalidade) são quase completos,
  enquanto gasto/saneamento "safely managed" têm missing alto.
- Vamos olhar se esse missing é aleatório ou depende da renda.

## 2. Missingness por grupo de renda (não aleatório)

In [3]:
miss = eda.missing_por_renda(abt)
miss.round(3)

,life_expectancy,child_mortality,uhc_index,uhc_sci,health_exp_per_capita,health_exp_gdp,out_of_pocket,gdp_per_capita,doctors_per_1000,nurses_per_1000,beds_per_1000,sanitation_basic,sanitation_safely,water_basic,water_safely,measles_imm_pct,dpt_imm_pct,urban_pct,fertility,population
Low income,0.0,0.000,0.180,0.294,0.360,0.360,0.360,0.107,0.574,0.661,0.728,0.318,0.522,0.315,0.633,0.049,0.049,0.0,0.0,0.0
Lower middle income,0.0,0.000,0.156,0.294,0.314,0.314,0.314,0.003,0.568,0.646,0.604,0.312,0.536,0.312,0.536,0.014,0.014,0.0,0.0,0.0
Upper middle income,0.0,0.000,0.165,0.306,0.316,0.314,0.314,0.016,0.421,0.484,0.399,0.326,0.554,0.327,0.539,0.036,0.036,0.0,0.0,0.0
High income,0.0,0.244,0.370,0.483,0.479,0.479,0.479,0.080,0.432,0.479,0.425,0.336,0.526,0.325,0.456,0.279,0.277,0.0,0.0,0.0


In [4]:
foco = ["doctors_per_1000", "health_exp_per_capita", "out_of_pocket",
        "sanitation_safely", "uhc_index"]
m = eda.missing_por_renda(abt).loc[:, [c for c in foco if c in miss.columns]].reset_index()
m = m.melt(id_vars="index", var_name="indicador", value_name="missing")
fig = px.bar(m, x="index", y="missing", color="indicador", barmode="group",
             labels={"index": "grupo de renda", "missing": "taxa de missing"},
             title="Missing por renda: países pobres têm menos dados")
fig

### Leitura

- O missing **cresce conforme a renda cai** para gasto, profissionais e saneamento:
  não é aleatório (MAR/MNAR), o que exige cuidado em imputação e no reporte do *n* efetivo.
- Isso justifica o **XGBoost com NaN nativo** em vez de imputar indiscriminadamente.

## 3. Distribuições e tendências

In [5]:
alvos = ["life_expectancy", "child_mortality"]
abt[alvos + ["uhc_index", "gdp_per_capita"]].describe().T

,count,mean,std,min,25%,50%,75%,max
life_expectancy,7378.0,69.479362,9.428367,12.158000,64.013750,71.394000,76.422738,86.372000
child_mortality,6664.0,45.308733,51.602141,1.300000,10.200000,23.500000,62.150000,489.300000
uhc_index,5564.0,0.507909,0.244034,0.010698,0.312493,0.525222,0.696414,0.973903
gdp_per_capita,7015.0,13590.035793,22471.748998,22.952133,1150.415528,4090.586174,17213.865371,256799.911212


In [6]:
long = abt.melt(id_vars=["year"], value_vars=alvos, var_name="indicador", value_name="valor")
fig = px.box(long, x="indicador", y="valor", color="indicador", points=False,
             title="Distribuição dos alvos (1990–2023)")
fig

In [7]:
serie = abt[abt["country_code"].isin(["BRA", "USA", "NGA", "IND", "JPN"])]
fig = px.line(serie, x="year", y="life_expectancy", color="country_code",
              title="Expectativa de vida — países selecionados")
fig

## 4. Correlações

In [8]:
corr = eda.correlacoes(abt)
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Correlação (Pearson) entre indicadores-chave")
fig

In [9]:
corr["life_expectancy"].drop("life_expectancy").sort_values()

child_mortality         -0.910827
fertility               -0.837270
out_of_pocket           -0.340712
population               0.005263
beds_per_1000            0.256693
health_exp_gdp           0.269376
gdp_per_capita           0.567538
nurses_per_1000          0.568936
health_exp_per_capita    0.575981
urban_pct                0.634621
measles_imm_pct          0.636384
doctors_per_1000         0.640732
dpt_imm_pct              0.657195
sanitation_safely        0.712347
water_basic              0.812958
uhc_index                0.839807
sanitation_basic         0.847125
water_safely             0.847334
uhc_sci                  0.854512
Name: life_expectancy, dtype: float64

### Leitura

- `life_expectancy` correlaciona forte e positivamente com saneamento, vacinação,
  médicos e `uhc_index`; e negativamente com `child_mortality` e fertilidade.
- Correlação ≠ causalidade: a direção do efeito UHC → expectativa de vida é testada
  no DiD (F7).

## 5. Comparativo regional

In [10]:
tend = eda.tendencias_regionais(abt, "life_expectancy")
fig = px.line(tend, x="year", y="life_expectancy", color="region",
              title="Expectativa de vida média por região")
fig

In [11]:
tend_cm = eda.tendencias_regionais(abt, "child_mortality")
fig = px.line(tend_cm, x="year", y="child_mortality", color="region",
              title="Mortalidade <5 média por região")
fig

## 6. Outliers (regra IQR 1.5×)

In [12]:
pd.DataFrame(eda.outliers_iqr(abt)).sort_values("outlier_rate", ascending=False)

,indicator,q1,q3,lower,upper,outliers,outlier_rate
4,health_exp_per_capita,64.678,8.995950e+02,-1.187698e+03,2.151970e+03,632,0.1393
19,population,630587.250,1.919320e+07,-2.721333e+07,4.703711e+07,898,0.1217
7,gdp_per_capita,1150.416,1.721387e+04,-2.294476e+04,4.130904e+04,657,0.0937
13,water_basic,80.832,9.932800e+01,5.308700e+01,1.270730e+02,404,0.0807
16,dpt_imm_pct,79.000,9.600000e+01,5.350000e+01,1.215000e+02,463,0.0720
1,child_mortality,10.200,6.215000e+01,-6.772500e+01,1.400750e+02,443,0.0665
15,measles_imm_pct,76.000,9.600000e+01,4.600000e+01,1.260000e+02,252,0.0392
10,beds_per_1000,1.600,5.525000e+00,-4.287000e+00,1.141200e+01,123,0.0328
9,nurses_per_1000,1.593,7.043000e+00,-6.582000e+00,1.521900e+01,56,0.0164
5,health_exp_gdp,4.139,7.965000e+00,-1.601000e+00,1.370500e+01,69,0.0152


Extremos de expectativa de vida (<20 anos) correspondem a eventos reais
(genocídio em Ruanda/1994, conflitos na RCA e Sudão do Sul), não a erros de digitação.

## 7. Validação do proxy UHC contra o SCI oficial

In [13]:
val = eda.validar_uhc(abt)
m = abt.dropna(subset=["uhc_index", "uhc_sci"])
fig = px.scatter(m, x="uhc_index", y="uhc_sci", color="region", opacity=0.4,
                 title=f"Proxy UHC vs SCI oficial (Pearson={val['pearson']}, "
                       f"Spearman={val['spearman']}, n={val['n']})")
fig

### Leitura

- O proxy (0..1) acompanha bem o SCI oficial (Pearson ≈ 0.90), mas **não substitui**
  a métrica oficial — serve para cobrir 1990–1999 e para o timing do DiD.
- Correlação alta valida a construção (pesos + normalização por percentis).

## Conclusões (conhecimento da base)

1. **Alvos confiáveis**: expectativa de vida e mortalidade <5 cobrem ~90%+ dos
   país-anos; são usáveis como outcomes dos modelos.
2. **Missing não aleatório**: piora com a renda → reportar *n* efetivo e usar modelo
   com suporte a NaN.
3. **UHC**: o proxy é consistente com o SCI oficial (r≈0.90); útil para 1990–2023 e
   para definir tratamento no DiD.
4. **Outliers reais**: conflitos explicam caudas extremas; não remover cegamente.
5. Próximo passo: visualizações (F4) e testes de hipótese (F5).